# Assignment No. 10 — Advanced NLP Techniques and Feature Engineering

In [1]:
# pip install nltk spacy scikit-learn
# python -m spacy download en_core_web_sm
import string
import pandas as pd
import nltk
import spacy
from nltk import pos_tag, word_tokenize, sent_tokenize, RegexpParser
from nltk.corpus import wordnet, stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

for p in ["punkt", "punkt_tab", "wordnet", "stopwords", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)

nlp = spacy.load("en_core_web_sm")

## Documents

In [2]:
docs = {
    "Doc1": "Apple released a new iPhone in California on Monday. The company said the phone has a faster chip and a better camera. Customers in London can buy it from Amazon next month.",
    "Doc2": "Samsung launched a new Galaxy phone in Seoul last week. The phone has a big screen and a good battery. Many reviewers in India liked the camera.",
    "Doc3": "The new Dell laptop is very fast and easy to use. The screen is bright and the battery lasts all day. I bought it in Lahore for my brother.",
}
for k, v in docs.items():
    print(k, ":", v)

Doc1 : Apple released a new iPhone in California on Monday. The company said the phone has a faster chip and a better camera. Customers in London can buy it from Amazon next month.
Doc2 : Samsung launched a new Galaxy phone in Seoul last week. The phone has a big screen and a good battery. Many reviewers in India liked the camera.
Doc3 : The new Dell laptop is very fast and easy to use. The screen is bright and the battery lasts all day. I bought it in Lahore for my brother.


## 1. POS tagging

In [3]:
for name, text in docs.items():
    print(name)
    print(pos_tag(word_tokenize(text)))
    print()

Doc1
[('Apple', 'NNP'), ('released', 'VBD'), ('a', 'DT'), ('new', 'JJ'), ('iPhone', 'NN'), ('in', 'IN'), ('California', 'NNP'), ('on', 'IN'), ('Monday', 'NNP'), ('.', '.'), ('The', 'DT'), ('company', 'NN'), ('said', 'VBD'), ('the', 'DT'), ('phone', 'NN'), ('has', 'VBZ'), ('a', 'DT'), ('faster', 'JJR'), ('chip', 'NN'), ('and', 'CC'), ('a', 'DT'), ('better', 'JJR'), ('camera', 'NN'), ('.', '.'), ('Customers', 'NNS'), ('in', 'IN'), ('London', 'NNP'), ('can', 'MD'), ('buy', 'VB'), ('it', 'PRP'), ('from', 'IN'), ('Amazon', 'NNP'), ('next', 'JJ'), ('month', 'NN'), ('.', '.')]

Doc2
[('Samsung', 'NNP'), ('launched', 'VBD'), ('a', 'DT'), ('new', 'JJ'), ('Galaxy', 'NNP'), ('phone', 'NN'), ('in', 'IN'), ('Seoul', 'NNP'), ('last', 'JJ'), ('week', 'NN'), ('.', '.'), ('The', 'DT'), ('phone', 'NN'), ('has', 'VBZ'), ('a', 'DT'), ('big', 'JJ'), ('screen', 'NN'), ('and', 'CC'), ('a', 'DT'), ('good', 'JJ'), ('battery', 'NN'), ('.', '.'), ('Many', 'JJ'), ('reviewers', 'NNS'), ('in', 'IN'), ('India', 'NNP

## 2. Named entity recognition (spaCy)

In [4]:
rows = []
for name, text in docs.items():
    for ent in nlp(text).ents:
        rows.append({"Document": name, "Entity": ent.text, "Label": ent.label_})
pd.DataFrame(rows)

,Document,Entity,Label
0,Doc1,Apple,ORG
1,Doc1,California,GPE
2,Doc1,Monday,DATE
3,Doc1,London,GPE
4,Doc1,Amazon,ORG
5,Doc1,next month,DATE
6,Doc2,Samsung,ORG
7,Doc2,Seoul,GPE
8,Doc2,last week,DATE
9,Doc2,India,GPE


## 3. Chunking and chinking

In [5]:
# Chunking: group determiner + adjectives + nouns into noun phrases (NP)
chunk_grammar = "NP: {<DT>?<JJ>*<NN.*>+}"
chunker = RegexpParser(chunk_grammar)

sentence = sent_tokenize(docs["Doc3"])[1]
print(sentence)
print(chunker.parse(pos_tag(word_tokenize(sentence))))

The screen is bright and the battery lasts all day.
(S
  (NP The/DT screen/NN)
  is/VBZ
  bright/JJ
  and/CC
  (NP the/DT battery/NN)
  lasts/VBZ
  (NP all/DT day/NN)
  ./.)


In [6]:
# Noun phrases found in every document
for name, text in docs.items():
    nps = []
    for s in sent_tokenize(text):
        tree = chunker.parse(pos_tag(word_tokenize(s)))
        nps += [" ".join(w for w, t in st.leaves()) for st in tree.subtrees() if st.label() == "NP"]
    print(name, nps)

Doc1 ['Apple', 'a new iPhone', 'California', 'Monday', 'The company', 'the phone', 'chip', 'camera', 'Customers', 'London', 'Amazon', 'next month']
Doc2 ['Samsung', 'a new Galaxy phone', 'Seoul', 'last week', 'The phone', 'a big screen', 'a good battery', 'Many reviewers', 'India', 'the camera']
Doc3 ['The new Dell laptop', 'The screen', 'the battery', 'all day', 'Lahore', 'brother']


In [7]:
# Chinking: take everything as a chunk, then remove (chink out) verbs, prepositions and conjunctions
chink_grammar = r"""
NP:
    {<.*>+}
    }<VB.*|IN|CC|\.>+{
"""
chinker = RegexpParser(chink_grammar)

sentence = sent_tokenize(docs["Doc1"])[0]
print(sentence)
print(chinker.parse(pos_tag(word_tokenize(sentence))))

Apple released a new iPhone in California on Monday.
(S
  (NP Apple/NNP)
  released/VBD
  (NP a/DT new/JJ iPhone/NN)
  in/IN
  (NP California/NNP)
  on/IN
  (NP Monday/NNP)
  ./.)


## 4. Lemmatization vs stemming

In [8]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = [("released", "v"), ("launched", "v"), ("customers", "n"), ("reviewers", "n"),
         ("faster", "a"), ("better", "a"), ("lasts", "v"), ("bought", "v"), ("liked", "v")]

pd.DataFrame({
    "Word": [w for w, _ in words],
    "Stem": [stemmer.stem(w) for w, _ in words],
    "Lemma": [lemmatizer.lemmatize(w, p) for w, p in words],
})

,Word,Stem,Lemma
0,released,releas,release
1,launched,launch,launch
2,customers,custom,customer
3,reviewers,review,reviewer
4,faster,faster,fast
5,better,better,good
6,lasts,last,last
7,bought,bought,buy
8,liked,like,like


Stemming just cuts the ending (`released` → `releas`, `bought` stays `bought`). Lemmatization gives real words (`bought` → `buy`, `better` → `good`).

## 5. WordNet synonyms and antonyms

In [9]:
for word in ["good", "fast", "happy", "big", "easy"]:
    synonyms, antonyms = set(), set()
    for syn in wordnet.synsets(word):
        for lem in syn.lemmas():
            synonyms.add(lem.name())
            for a in lem.antonyms():
                antonyms.add(a.name())
    synonyms.discard(word)
    print(word)
    print("  Synonyms:", sorted(synonyms)[:8])
    print("  Antonyms:", sorted(antonyms))

good
  Synonyms: ['adept', 'beneficial', 'commodity', 'dear', 'dependable', 'effective', 'estimable', 'expert']
  Antonyms: ['bad', 'badness', 'evil', 'evilness', 'ill']
fast
  Synonyms: ['debauched', 'degenerate', 'degraded', 'dissipated', 'dissolute', 'fasting', 'firm', 'flying']
  Antonyms: ['slow']
happy
  Synonyms: ['felicitous', 'glad', 'well-chosen']
  Antonyms: ['unhappy']
big
  Synonyms: ['adult', 'bad', 'bighearted', 'boastful', 'boastfully', 'bounteous', 'bountiful', 'braggart']
  Antonyms: ['little', 'small']
easy
  Synonyms: ['comfortable', 'easily', 'easygoing', 'gentle', 'leisurely', 'light', 'loose', 'promiscuous']
  Antonyms: ['difficult', 'quickly', 'uneasy']


WordNet returns synonyms for every meaning of a word, so some of them do not fit our context (for example `big` gives `bad`).

## 6. Bag of Words

In [10]:
cv = CountVectorizer(stop_words="english")
bow = cv.fit_transform(docs.values())

bow_df = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=docs.keys())
print("Shape (documents x words):", bow_df.shape)
bow_df

Shape (documents x words): (3, 41)


,amazon,apple,battery,better,big,bought,bright,brother,buy,california,...,new,phone,released,reviewers,said,samsung,screen,seoul,use,week
Doc1,1,1,0,1,0,0,0,0,1,1,...,1,1,1,0,1,0,0,0,0,0
Doc2,0,0,1,0,1,0,0,0,0,0,...,1,2,0,1,0,1,1,1,0,1
Doc3,0,0,1,0,0,1,1,1,0,0,...,1,0,0,0,0,0,1,0,1,0


## 7. Feature extraction (TF-IDF + PCA)

In [11]:
# TF-IDF gives weights instead of raw counts, then PCA reduces the many word features to 2 components
tfidf = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf.fit_transform(docs.values())

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf.toarray())

print("Before:", X_tfidf.shape, " After:", X_pca.shape)
print("Variance explained:", pca.explained_variance_ratio_.round(3))
pd.DataFrame(X_pca, columns=["Component 1", "Component 2"], index=docs.keys()).round(3)

Before: (3, 41)  After: (3, 2)
Variance explained: [0.537 0.463]


,Component 1,Component 2
Doc1,-0.656,-0.437
Doc2,-0.080,0.746
Doc3,0.735,-0.309


## 8. Document similarity (cosine similarity)

In [12]:
sim = cosine_similarity(X_tfidf)
sim_df = pd.DataFrame(sim, index=docs.keys(), columns=docs.keys()).round(3)
print(sim_df)
print()
print("Doc1 vs Doc2 :", sim_df.loc["Doc1", "Doc2"])
print("Doc1 vs Doc3 :", sim_df.loc["Doc1", "Doc3"])
print("Doc2 vs Doc3 :", sim_df.loc["Doc2", "Doc3"])

       Doc1   Doc2   Doc3
Doc1  1.000  0.135  0.024
Doc2  0.135  1.000  0.112
Doc3  0.024  0.112  1.000

Doc1 vs Doc2 : 0.135
Doc1 vs Doc3 : 0.024
Doc2 vs Doc3 : 0.112


Doc1 and Doc2 are the most similar pair (both about new phones) and Doc1 and Doc3 the least. All scores are low because the documents are short and share very few words once stopwords are removed.